# Comparing Model Providers in MemoRizz

MemoRizz supports multiple LLM providers through a unified interface. This notebook
demonstrates how to switch between providers with minimal code changes.

**Providers covered:**

| Provider | Type | Key Models | Best For |
|----------|------|------------|----------|
| **OpenAI** | Cloud API | GPT-4o, GPT-4o Mini | General purpose, proven ecosystem |
| **Anthropic** | Cloud API | Claude Sonnet, Opus, Haiku | Nuanced reasoning, safety, long context |
| **Azure OpenAI** | Cloud API | Same as OpenAI (managed) | Enterprise deployments |
| **Ollama** | Local | Llama 3, Mistral, Gemma | Privacy, no API costs, offline use |
| **HuggingFace** | Local | Any HF model | Custom/fine-tuned models |

> **Note:** You only need API keys for the providers you want to use.

In [ ]:
%pip install -qU memorizz

In [ ]:
import os
import getpass

# Configure whichever API keys you have
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (or Enter to skip): ") or "skip"

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (or Enter to skip): ") or "skip"

print("Keys configured.")

---
## The Power of Config Dicts

Switching providers is as simple as changing the `"provider"` and `"model"` keys.
The rest of your code stays exactly the same.

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder

# Define configs for each provider
PROVIDER_CONFIGS = {
    "openai": {
        "provider": "openai",
        "model": "gpt-4o-mini",
    },
    "anthropic": {
        "provider": "anthropic",
        "model": "claude-sonnet-4-5-20250929",
    },
    # Uncomment if you have Ollama running:
    # "ollama": {
    #     "provider": "ollama",
    #     "model": "llama3.1",
    # },
}

prompt = "In exactly one sentence, explain what a neural network is."

for name, config in PROVIDER_CONFIGS.items():
    if os.environ.get(f"{name.upper()}_API_KEY", "") == "skip":
        print(f"\n{name}: (skipped — no API key)")
        continue
    try:
        agent = (
            MemAgentBuilder()
            .with_instruction("Be concise.")
            .with_llm_config(config)
            .build()
        )
        response = agent.run(prompt)
        print(f"\n{name} ({config['model']}):\n{response}")
    except Exception as e:
        print(f"\n{name}: Error — {e}")

---
## Quick-Start Config Templates

Copy-paste these configs into your project:

In [ ]:
# OpenAI
openai_config = {
    "provider": "openai",
    "model": "gpt-4o",        # or "gpt-4o-mini" for lower cost
    "temperature": 0.7,
}

# Anthropic
anthropic_config = {
    "provider": "anthropic",
    "model": "claude-sonnet-4-5-20250929",
    "max_tokens": 4096,
}

# Azure OpenAI
azure_config = {
    "provider": "azure",
    "deployment_name": "my-gpt4-deployment",
    "azure_endpoint": "https://my-resource.openai.azure.com/",
    "api_version": "2024-06-01",
}

# Ollama (local)
ollama_config = {
    "provider": "ollama",
    "model": "llama3.1",       # Must be pulled: ollama pull llama3.1
    "host": "http://localhost:11434",
}

# HuggingFace (local)
huggingface_config = {
    "provider": "huggingface",
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "device": "cuda",          # or "cpu", "mps"
}

print("Config templates ready. Use any of these with:\n")
print('  MemAgentBuilder().with_llm_config(config).build()')

---
## Feature Comparison

| Feature | OpenAI | Anthropic | Azure | Ollama | HuggingFace |
|---------|--------|-----------|-------|--------|-------------|
| Tool calling | Yes | Yes | Yes | Yes* | No |
| Streaming | Yes | Yes | No | Yes | No |
| API key required | Yes | Yes | Yes | No | No** |
| Runs locally | No | No | No | Yes | Yes |
| Context window | Up to 1M | 200K | Varies | Varies | Varies |
| Cost | Per token | Per token | Per token | Free | Free |

\* Tool calling requires compatible models (Llama 3.1+, Qwen 2.5+, Mistral)
\** HuggingFace token needed only for private/gated models

### Choosing a Provider

- **Best quality:** OpenAI GPT-4o or Anthropic Claude Sonnet 4.5
- **Best cost:** OpenAI GPT-4o Mini or Anthropic Claude Haiku 4.5
- **Best privacy:** Ollama (everything stays on your machine)
- **Enterprise:** Azure OpenAI (SLAs, compliance, managed infrastructure)
- **Custom models:** HuggingFace (fine-tuned models, specialized architectures)